In [9]:
import io
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as et

In [10]:
##import docker em XML
endpoint = 'https://divvy-tripdata.s3.amazonaws.com/?list-type=2'
response = requests.get(endpoint)
print(response.text)


<?xml version="1.0" encoding="UTF-8"?>
<ListBucketResult xmlns="http://s3.amazonaws.com/doc/2006-03-01/"><Name>divvy-tripdata</Name><Prefix></Prefix><KeyCount>95</KeyCount><MaxKeys>1000</MaxKeys><IsTruncated>false</IsTruncated><Contents><Key>202004-divvy-tripdata.zip</Key><LastModified>2020-06-01T14:50:06.000Z</LastModified><ETag>&quot;e7a221ace4629d53dcd73a62b314b567&quot;</ETag><Size>3323572</Size><StorageClass>STANDARD</StorageClass></Contents><Contents><Key>202005-divvy-tripdata.zip</Key><LastModified>2020-06-01T14:50:09.000Z</LastModified><ETag>&quot;606a191a00a58840ce8d3cf7d08556e4&quot;</ETag><Size>7988821</Size><StorageClass>STANDARD</StorageClass></Contents><Contents><Key>202006-divvy-tripdata.zip</Key><LastModified>2020-07-06T00:31:49.000Z</LastModified><ETag>&quot;e397c3a64e4f8d4cefe90736cce330eb&quot;</ETag><Size>14732088</Size><StorageClass>STANDARD</StorageClass></Contents><Contents><Key>202007-divvy-tripdata.zip</Key><LastModified>2020-08-12T02:10:49.000Z</LastModified><

In [11]:
##transformar em arq virtual
ns = {'s3': 'http://s3.amazonaws.com/doc/2006-03-01/'}
raiz = et.fromstring(response.content) 
elementos_contents = raiz.findall('s3:Contents', ns)
print(elementos_contents)

[<Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293E9B67E70>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A1F9BF10>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A1F64860>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A1F66020>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A1F66430>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A718F060>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A718D9E0>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A718ECF0>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A718F8D0>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A718E6B0>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 0x00000293A718DCB0>, <Element '{http://s3.amazonaws.com/doc/2006-03-01/}Contents' at 

In [12]:
##loop de insercao dos arqs
arquivos_zip = []
for elemento in elementos_contents:
    chave = elemento.find('s3:Key', ns)

    if chave is not None:
        nome_arq = chave.text

        if nome_arq.endswith(".zip") and "tripdata" in nome_arq.lower():
            arquivos_zip.append(nome_arq)
print(arquivos_zip)

['202004-divvy-tripdata.zip', '202005-divvy-tripdata.zip', '202006-divvy-tripdata.zip', '202007-divvy-tripdata.zip', '202008-divvy-tripdata.zip', '202009-divvy-tripdata.zip', '202010-divvy-tripdata.zip', '202011-divvy-tripdata.zip', '202012-divvy-tripdata.zip', '202101-divvy-tripdata.zip', '202102-divvy-tripdata.zip', '202103-divvy-tripdata.zip', '202104-divvy-tripdata.zip', '202105-divvy-tripdata.zip', '202106-divvy-tripdata.zip', '202107-divvy-tripdata.zip', '202108-divvy-tripdata.zip', '202109-divvy-tripdata.zip', '202110-divvy-tripdata.zip', '202111-divvy-tripdata.zip', '202112-divvy-tripdata.zip', '202201-divvy-tripdata.zip', '202202-divvy-tripdata.zip', '202203-divvy-tripdata.zip', '202204-divvy-tripdata.zip', '202205-divvy-tripdata.zip', '202206-divvy-tripdata.zip', '202207-divvy-tripdata.zip', '202208-divvy-tripdata.zip', '202209-divvy-tripdata.zip', '202210-divvy-tripdata.zip', '202211-divvy-tripdata.zip', '202212-divvy-tripdata.zip', '202301-divvy-tripdata.zip', '202302-divvy

In [ ]:
##verificar a usabilidade de converter o df em parquet, para reduzir o tamanho do arquivo e melhorar a performance de leitura e escrita.
lista_dataframes = []
##comente o [:3]: para rodar toda a base do S3:
for nome_zip in arquivos_zip:##[:3]:
    print(f"Processando arquivo: {nome_zip}")

    url_zip = f"https://divvy-tripdata.s3.amazonaws.com/{nome_zip}"

    respota_zip = requests.get(url_zip)

    buffer_zip = io.BytesIO(respota_zip.content)

    with zipfile.ZipFile(buffer_zip) as arquivo_zip:

        nome_csv = [f for f in arquivo_zip.namelist() if f.endswith('.csv')]

        if nome_csv:
            arquivo_csv = nome_csv[0]

            with arquivo_zip.open(arquivo_csv) as csv_file:
                df = pd.read_csv(
                    csv_file, 
                    encoding='latin-1',
                    low_memory=False,
                    dtype={
                    'start_station_id': 'str',
                    'end_station_id': 'str',
                    'ride_id': 'str'
                    }
                )
                lista_dataframes.append(df)
        else:
            print(f"Nenhum arquivo CSV encontrado no arquivo ZIP: {nome_zip}")

print(f"Total de DataFrames processados: {len(lista_dataframes)}")

df_concatenado = pd.concat(lista_dataframes, ignore_index=True)

df_concatenado.to_parquet("divvy_tripdata_consolidado.parquet", index=False)

df = pd.read_parquet("divvy_tripdata_consolidado.parquet")

df.head()


Processando arquivo: 202004-divvy-tripdata.zip
Processando arquivo: 202005-divvy-tripdata.zip
Processando arquivo: 202006-divvy-tripdata.zip
Processando arquivo: 202007-divvy-tripdata.zip
Processando arquivo: 202008-divvy-tripdata.zip
Processando arquivo: 202009-divvy-tripdata.zip
Processando arquivo: 202010-divvy-tripdata.zip
Processando arquivo: 202011-divvy-tripdata.zip
Processando arquivo: 202012-divvy-tripdata.zip
Processando arquivo: 202101-divvy-tripdata.zip
Processando arquivo: 202102-divvy-tripdata.zip
Processando arquivo: 202103-divvy-tripdata.zip
Processando arquivo: 202104-divvy-tripdata.zip
Processando arquivo: 202105-divvy-tripdata.zip
Processando arquivo: 202106-divvy-tripdata.zip
Processando arquivo: 202107-divvy-tripdata.zip
Processando arquivo: 202108-divvy-tripdata.zip
Processando arquivo: 202109-divvy-tripdata.zip
Processando arquivo: 202110-divvy-tripdata.zip
Processando arquivo: 202111-divvy-tripdata.zip
Processando arquivo: 202112-divvy-tripdata.zip
Processando a